### **Step 1: Request files from CartoSpot API and read them into Dataframes**
#### Enter the ID to get the required INTERVAL Dataset after running the cell:

In [1]:
import os
import sys
import zipfile
from io import BytesIO
from pathlib import Path
import requests
import pandas as pd
import numpy as np
import geojson
import geopandas as gpd
from shapely.geometry import Point

ROOT = Path.cwd().resolve()
if ROOT.name == "src":
    ROOT = ROOT.parent
OUTPUTS = ROOT / "outputs"

# Get the data Id from user input
print("----")
data_id = input("Enter the data Id for your INTERVAL Dataset (e.g., 306): ")
# data_id = 306

# Validate user input (ensure it's a positive integer)
try:
    data_id = int(data_id)
    if data_id <= 0:
        raise ValueError("Data Id must be a positive integer.")
except ValueError as e:
    print(f"Invalid input: {e}")
    sys.exit(1)  # Exit with error code 1

# Define the API endpoint and file paths
api_endpoint = f"https://api.cartospot.com/api/v2/db/cartospot/generate-csv/interval/{data_id}/"
csv_folder = str(OUTPUTS / "csv_files")
places_file_path = os.path.join(csv_folder, "places.csv")
submissions_file_path = os.path.join(csv_folder, "submissions.csv")

# Ensure the folder for CSV files exists
if not os.path.exists(csv_folder):
    os.makedirs(csv_folder)

# Request the API endpoint and extract files
try:
    response = requests.get(api_endpoint)
    if response.status_code == 200:
        # Assume the response is a ZIP file
        with zipfile.ZipFile(BytesIO(response.content)) as z:
            z.extractall(csv_folder)
        print("----")
        print(f"Extracted all files to '{csv_folder}'.")

        # Read the extracted CSV files into DataFrames
        try:
            places_df = pd.read_csv(places_file_path)
            submissions_df = pd.read_csv(submissions_file_path)
            print("----")
            print("CSV files loaded into DataFrames successfully!")
        except Exception as e:
            print(f"Unexpected error reading CSV files: {e}")
            sys.exit(2)  # Exit with error code 2
    else:
        print(f"Failed to download files. Status code: {response.status_code}")
        sys.exit(3)  # Exit with error code 3
except Exception as e:
    print(f"Error fetching and extracting files: {e}")
    sys.exit(4)  # Exit with error code 4

----


Enter the data Id for your INTERVAL Dataset (e.g., 306):  301


----
Extracted all files to 'outputs/csv_files'.
----
CSV files loaded into DataFrames successfully!


### **Step 2: Preview Places and Submissions raw data**
#### Run cell below:

In [39]:
# View Places dataframe data types
print(places_df.info())
# View first Places 10 rows
print(places_df.head(10))

# View Submissions dataframe data types
print(submissions_df.info())
# View first 10 Submissions rows
print(submissions_df.head(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1023 entries, 0 to 1022
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Id                1023 non-null   int64  
 1   unique_id         869 non-null    object 
 2   location_type     1023 non-null   object 
 3   tree_privacy      1023 non-null   object 
 4   NTM_ID            0 non-null      float64
 5   tree_height       798 non-null    float64
 6   avg_tree_height   798 non-null    float64
 7   canopy_perimeter  798 non-null    float64
 8   canopy_area       798 non-null    float64
 9   STD               798 non-null    float64
 10  geometry_x        1023 non-null   float64
 11  geometry_y        1023 non-null   float64
 12  dataset_id        1023 non-null   int64  
 13  dataset_name      1023 non-null   object 
 14  url               1023 non-null   object 
 15  submitter_name    949 non-null    object 
 16  submitter_id      867 non-null    float64


### **Step 3: Clean and merge data to export final CSVs, GeoJSON and SHP files**
#### Run the cell below:

In [2]:
# Replace 'Other' in species with other_species data if applicable
submissions_df['species'] = np.where(
    (submissions_df['species'] == 'Other') & (submissions_df['other_species'].notna()),
    submissions_df['other_species'],
    submissions_df['species']
)

# Reduce and rename columns for Places DataFrame here
places_df_trans = places_df[
    ['Id', 'location_type', 'tree_privacy', 'NTM_ID', 'tree_height', 'avg_tree_height', 'canopy_perimeter', 'canopy_area', 'geometry_x', 'geometry_y', 'dataset_id', 'dataset_name', 'url']
].rename(columns={
    'Id': 'location_id',
    'url': 'location_url'
})

# Reduce and rename columns for Submissions DataFrame here
submissions_df_trans = submissions_df[
    ['Id', 'location_id', 'does_a_tree_exist_here', 'species', 'tree_type', 'health', 'circumference', 'dbh', 'notes', 'image_url', 'submitter_name']
].rename(columns={
    'Id': 'submission_id'
})

# Merge the transformed DataFrames (preserve all places with how='left')
merged_df = pd.merge(
    places_df_trans,
    submissions_df_trans,
    on='location_id',
    how='left'
)

# Split into one-to-many and one-to-zero-or-one DataFrames
multiple_submissions_df = merged_df[merged_df.duplicated(subset='location_id', keep=False)]
valid_submissions_df = merged_df[~merged_df.duplicated(subset='location_id', keep=False)]

# Save the DataFrames to CSV files
output_folder_csv = str(OUTPUTS / "csv_files")
os.makedirs(output_folder_csv, exist_ok=True)

data_multiple_file = os.path.join(output_folder_csv, f"data_multiple_submissions_dataset_id_{data_id}.csv")
data_valid_file = os.path.join(output_folder_csv, f"data_valid_submissions_dataset_id_{data_id}.csv")

multiple_submissions_df.to_csv(data_multiple_file, index=False)
valid_submissions_df.to_csv(data_valid_file, index=False)

print(f"Saved multiple submissions to '{data_multiple_file}'.")
print("----")
print(f"Saved valid submissions to '{data_valid_file}'.")

# Create GeoJSON file
output_folder_geofiles = str(OUTPUTS / "geofiles")
os.makedirs(output_folder_geofiles, exist_ok=True)

geojson_file = os.path.join(output_folder_geofiles, f"data_valid_submissions_dataset_id_{data_id}.geojson")
features = [
    geojson.Feature(
        id=row['location_id'],
        geometry=geojson.Point((row['geometry_x'], row['geometry_y'])),
        properties=row.drop(['geometry_x', 'geometry_y', 'location_id']).to_dict()
    )
    for _, row in valid_submissions_df.fillna('').iterrows()
]

geojson_data = geojson.FeatureCollection(features)
with open(geojson_file, 'w') as f:
    geojson.dump(geojson_data, f)
print("----")
print(f"GeoJSON file created: {geojson_file}")

# Create Shapefile
shapefile_path = os.path.join(output_folder_geofiles, f"data_valid_submissions_dataset_id_{data_id}.shp")
gdf = gpd.GeoDataFrame(
    valid_submissions_df.fillna(''),
    geometry=[Point(xy) for xy in zip(valid_submissions_df['geometry_x'], valid_submissions_df['geometry_y'])],
    crs="EPSG:4326"
).drop(columns=['geometry_x', 'geometry_y']).rename(columns={
    'location_id': 'loc_id',
    'location_type': 'loc_type',
    'tree_privacy': 'privacy',
    'tree_height': 'height',
    'avg_tree_height': 'avg_height',
    'canopy_perimeter': 'canopy_per',
    'canopy_area': 'canopy_are',
    'dataset_name': 'dataset',
    'location_url': 'loc_url',
    'submission_id': 'sub_id',
    'does_a_tree_exist_here': 'tree_exist',
    'circumference': 'circumf',
    'submitter_name': 'submitter'
})
gdf.to_file(shapefile_path, driver="ESRI Shapefile")
print("----")
print(f"Shapefile created: {shapefile_path}")

Saved multiple submissions to 'outputs/csv_files/data_multiple_submissions_dataset_id_301.csv'.
----
Saved valid submissions to 'outputs/csv_files/data_valid_submissions_dataset_id_301.csv'.
----
GeoJSON file created: outputs/geofiles/data_valid_submissions_dataset_id_301.geojson
----
Shapefile created: outputs/geofiles/data_valid_submissions_dataset_id_301.shp


### **Step 4: Run cell below to generate a ZIP with all the files for the dataset**

In [3]:
# Define the ZIP file path
zip_file_path = str(OUTPUTS / f"dataset_id_{data_id}_outputs.zip")
OUTPUTS.mkdir(parents=True, exist_ok=True)

# Prepare files to include in the ZIP
files_to_zip = [
    (places_file_path, "places.csv"),  # Initial places file
    (submissions_file_path, "submissions.csv"),  # Initial submissions file
    (data_multiple_file, f"data_multiple_submissions_dataset_id_{data_id}.csv"),  # Processed CSV
    (data_valid_file, f"data_valid_submissions_dataset_id_{data_id}.csv"),  # Processed CSV
    (geojson_file, f"data_valid_submissions_dataset_id_{data_id}.geojson")  # GeoJSON
]

# Add only the Shapefile components (explicit extensions for .shp, .shx, .prj, .dbf, .cpg)
shapefile_base = Path(shapefile_path).stem  # Get base name of the shapefile
shapefile_dir = Path(shapefile_path).parent
shapefile_extensions = ['.shp', '.shx', '.prj', '.dbf', '.cpg']  # Expected Shapefile components
shapefile_components = [
    shapefile_dir / f"{shapefile_base}{ext}" for ext in shapefile_extensions if (shapefile_dir / f"{shapefile_base}{ext}").exists()
]

files_to_zip.extend([(str(file), file.name) for file in shapefile_components])

# Create the ZIP file
added_files = set()  # Track files already added to avoid duplicates
with zipfile.ZipFile(zip_file_path, 'w') as zipf:
    for file_path, arcname in files_to_zip:
        if arcname not in added_files:  # Avoid duplicate additions
            zipf.write(file_path, arcname=arcname)
            added_files.add(arcname)

print("----")
print(f"ZIP file created: {zip_file_path}")

----
ZIP file created: outputs/dataset_id_301_outputs.zip
